# gradb2: which sparkle version is in each global store

One figure. Every path is written out literally below — nothing is read from a
config, and no other file needs opening to know where a number came from.

| what | where |
|---|---|
| your global fields (tiles) | `s3://dbof/globals_for_chunks/V5/20120704_120000/frontal_structure.zarr` |
| your colleague's globals | `s3://dbof/globals_for_cutouts/v2_2_01/20120704_120000/frontal_structure.zarr` |
| the raw chunk, one snapshot | `s3://dbof/LLC4320_RAW/CHUNKS/monterey_bay/20120704T12.zarr` |
| the raw chunk's grid | `s3://dbof/LLC4320_RAW/CHUNKS/monterey_bay/grid.zarr` |

The live tile is the reference. `dbof.preprocessing.calculate_fields.grad_b2` is
applied to the chunk on the spot — the same function `surface_subsets` maps
`"gradb2"` to, so it is the pipeline's own code on whatever `dbof` is installed
now. Nothing is written to disk. Whichever store matches it is current.

## The sparkle

At a one-cell gradient extremum the slopes on either side are `-a` and `+a`.
Interpolating them to the cell centre averages them to **zero**, and squaring
locks that in. The old form therefore *manufactures near-zeros* — holes — at
exactly the grid-scale features that matter. The fix squares on the staggered
points first, keeping the real value.

Per direction the two forms differ by an exact identity:

```
new - old  =  1/4 (g1 - g2)^2
```

so the difference is **never negative**, and is a map of grid-scale gradient
curvature: zero in smooth water, bright along filaments. Blue in a
`new - old` panel would mean something other than the formula is going on.

## Provenance

| | |
|---|---|
| fix committed | `88fcf1d` 2026-08-06 23:04 -0700 |
| merged to `main` | `a3e43eb` 2026-08-10 07:52 -0700 (PR #31 `gradient-sparkles`) |
| `globals_for_chunks/V5` | current form — matches the live tile |
| `globals_for_cutouts/v2_2_01` | superseded form; chunks written 2026-08-10 18:16 UTC |

`v2_2_01/run_meta.yaml` records `git_commit: 48f5031` (2026-08-10 07:58 -0700),
which *does* contain the fix — but `_git_commit_hash()` in
`dbof.global_dataset_creation.metadata` runs `git rev-parse HEAD` with no `cwd`
and no reference to `dbof.__file__`. It records the repo the shell was standing
in, not the package that was imported, so it cannot testify to what ran. The
measurement below is the evidence, not the metadata.

Channels affected by the fix: `gradb2`, `gradtheta2`, `gradsalt2`, `gradrho2`,
`gradeta2`, `strain_mag`, `okubo_weiss`. Not `relative_vorticity`,
`divergence`, `rossby_number`, `turner_angle`, `density`, `buoyancy`,
`frontogenesis_tendency`.

In [ ]:
%matplotlib inline
import fsspec
import numpy as np
import xarray as xr
import zarr
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from dbof.preprocessing import calculate_fields as cf   # grad_b2 itself
from dbof.tiles import tile_utils                       # builds the xgcm grid
from fronts.llc import tiles as llc_tiles               # rect <-> face index maps

# ---------------------------------------------------------------------------
# Everything this notebook reads.
# ---------------------------------------------------------------------------
S3_ENDPOINT   = 'https://s3-west.nrp-nautilus.io'

GLOBAL_CHUNKS  = 's3://dbof/globals_for_chunks/V5/20120704_120000/frontal_structure.zarr'
GLOBAL_CUTOUTS = 's3://dbof/globals_for_cutouts/v2_2_01/20120704_120000/frontal_structure.zarr'
CHUNK_DATA     = 's3://dbof/LLC4320_RAW/CHUNKS/monterey_bay/20120704T12.zarr'
CHUNK_GRID     = 's3://dbof/LLC4320_RAW/CHUNKS/monterey_bay/grid.zarr'

CHANNEL = 'gradb2'
BOX_KM  = 200
TITLE   = 'gradb2  2012-07-04 12:00  monterey_bay'

# Path addressing and s3v4 signing are what this endpoint needs.
fs = fsspec.filesystem(
    's3', asynchronous=False,
    client_kwargs={'endpoint_url': S3_ENDPOINT},
    config_kwargs={'signature_version': 's3v4',
                   'request_checksum_calculation': 'when_required',
                   's3': {'addressing_style': 'path',
                          'payload_signing_enabled': False,
                          'use_accelerate_endpoint': False,
                          'use_dualstack_endpoint': False}})

def open_chunk_store(uri):
    """Open one of the raw CHUNKS stores lazily.  Metadata is not consolidated."""
    return xr.open_zarr(uri, consolidated=False,
                        storage_options={'client_kwargs':
                                         {'endpoint_url': S3_ENDPOINT}})

def read_global_channel(uri, channel):
    """Pull one named channel out of a global snapshot store -> (12960, 17280)."""
    root = zarr.open_group(store=zarr.storage.FsspecStore(path=uri, fs=fs),
                           mode='r', use_consolidated=False)
    idx = list(root.attrs['channel_names']).index(channel)
    return np.asarray(root['data'][idx], dtype=np.float32)

# Sequential: one hue light->dark for magnitude.  Diverging: blue<->red with a
# neutral gray midpoint so "no difference" reads as nothing.
SEQ = 'Blues'
DIV = LinearSegmentedColormap.from_list(
    'blue_gray_red', ['#184f95', '#86b6ef', '#f0efec', '#f0a3a2', '#a52322'])

import matplotlib
if 'inline' not in matplotlib.get_backend():
    try:
        plt.switch_backend('module://matplotlib_inline.backend_inline')
    except Exception as exc:
        print(f'could not switch off {matplotlib.get_backend()}: {exc}\n'
              'figures will not render -- pip install matplotlib-inline '
              'and restart the kernel')
print('matplotlib backend:', matplotlib.get_backend())

In [ ]:
# --- the chunk: which 720x720 tile is it, and how big is a 200 km box? ------
ds_grid = open_chunk_store(CHUNK_GRID)

# xgcm locates its axes from comodo attrs, which the chunk transfer does not
# copy across.  Without them Grid() has no X or Y.  The shift sign decides
# which dxC a staggered difference is paired with, so it is load-bearing: get it
# wrong and every gradient is off by one cell's metric while pointwise fields
# stay exact.  +0.5 is this dataset's convention, from
# dbof.llc4320_ingestion.get_raw_data.get_llc_depth_gridfile.
COMODO = {
    'j':   {'axis': 'Y'},
    'j_g': {'axis': 'Y', 'c_grid_axis_shift': 0.5},
    'i':   {'axis': 'X'},
    'i_g': {'axis': 'X', 'c_grid_axis_shift': 0.5},
}
for dim, attrs in COMODO.items():
    if dim not in ds_grid.coords:
        raise KeyError(f'{dim} is not a coordinate of {CHUNK_GRID}; '
                       'xgcm cannot be given an axis for it')
    ds_grid[dim].attrs.update(attrs)

face = int(ds_grid.attrs['resolved_face'])
j0   = int(ds_grid.attrs['j_start'])
i0   = int(ds_grid.attrs['i_start'])

face_id, j_map, i_map = llc_tiles.lookup_maps()
j_rect, i_rect = np.argwhere((face_id == face) & (j_map == j0) & (i_map == i0))[0]
tile = llc_tiles.rect_ij_to_tile(int(i_rect), int(j_rect))
print(f'{CHUNK_GRID}\n    face {face}, face-local j={j0} i={i0}  ->  '
      f'tile {tile.tile_idx}, rect j={tile.rect_j_slice.start} '
      f'i={tile.rect_i_slice.start}')

dx_km = float(np.nanmean(ds_grid['dxC'].values)) / 1e3
half  = int(BOX_KM / dx_km / 2)
c     = llc_tiles.TILE_SIZE // 2
box   = (slice(c - half, c + half), slice(c - half, c + half))
print(f'    dxC {dx_km:.2f} km  ->  {2*half} x {2*half} cells '
      f'= {2*half*dx_km:.0f} km across\n')

# --- gradb2 from each source ----------------------------------------------
fields = {}
for label, uri in (('globals_for_chunks/V5',        GLOBAL_CHUNKS),
                   ('globals_for_cutouts/v2_2_01',  GLOBAL_CUTOUTS)):
    g = read_global_channel(uri, CHANNEL)
    # The global map is on the rect grid; the chunk computes in face-local
    # (j, i), a rotation of it on some faces.  Reorient, then crop.
    fields[label] = llc_tiles.labels_for_tile(g, tile)[box]
    del g
    print(f'read {label}')

ds_snap = open_chunk_store(CHUNK_DATA)
if 'time' in ds_snap.dims:
    ds_snap = ds_snap.isel(time=0)
ds_merge, xgrid = tile_utils._build_tile_context(ds_snap, ds_grid)
fields['tile, computed live'] = llc_tiles.surface(cf.grad_b2(ds_merge, xgrid))[box]
print('computed grad_b2 live on the chunk')

# --- how far apart are they? ---------------------------------------------
ref   = 'tile, computed live'
pairs = [(k, ref) for k in fields if k != ref]
pairs.append(('globals_for_chunks/V5', 'globals_for_cutouts/v2_2_01'))

print()
for a, b in pairs:
    A, B = fields[a], fields[b]
    m = np.isfinite(A) & np.isfinite(B) & (B > 0)
    rel = np.abs(A[m] - B[m]) / B[m]
    print(f'{a} vs {b}\n    median |rel| {np.median(rel):.2e}   '
          f'p95 {np.percentile(rel, 95):.2e}')

In [ ]:
logs  = {k: np.log10(np.where(v > 0, v, np.nan)) for k, v in fields.items()}
diffs = {f'{a}\n$-$ {b}': logs[a] - logs[b] for a, b in pairs}
vmin, vmax = np.nanpercentile(logs[ref], [2, 98])
lim = max(float(np.nanpercentile(np.abs(d), 99)) for d in diffs.values())

fig, ax = plt.subplots(2, 3, figsize=(16, 9.2))
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values():
        s.set_color('#b8b6b0')

for col, (name, arr) in enumerate(logs.items()):
    im = ax[0, col].imshow(arr, origin='lower', cmap=SEQ, vmin=vmin, vmax=vmax)
    ax[0, col].set_title(name, fontsize=10)
    fig.colorbar(im, ax=ax[0, col], fraction=0.046, label='log$_{10}$ gradb2')

for col, (name, arr) in enumerate(diffs.items()):
    im = ax[1, col].imshow(arr, origin='lower', cmap=DIV, vmin=-lim, vmax=lim)
    ax[1, col].set_title(name, fontsize=10)
    fig.colorbar(im, ax=ax[1, col], fraction=0.046, label='dex')

fig.suptitle(f'{TITLE}  —  {2*half*dx_km:.0f} km box', fontsize=12)
fig.tight_layout()
plt.show()